In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression

In [2]:
ds = xr.open_dataset("/home/dtaneja/analysis-dishika/notebooks/HRDPS_2007_pressure.nc")
print(ds)

<xarray.Dataset> Size: 791MB
Dimensions:       (time_counter: 2904, y: 266, x: 256)
Coordinates:
  * time_counter  (time_counter) datetime64[ns] 23kB 2007-01-03 ... 2007-12-3...
Dimensions without coordinates: y, x
Data variables:
    pressure      (time_counter, y, x) float32 791MB ...


In [4]:
print(ds.time_counter.values[0])
print(ds.time_counter.values[1])
print(ds.time_counter.values[2])
print(ds.time_counter.values[-3])
print(ds.time_counter.values[-2])
print(ds.time_counter.values[-1])

2007-01-03T00:00:00.000000000
2007-01-03T03:00:00.000000000
2007-01-03T06:00:00.000000000
2007-12-31T15:00:00.000000000
2007-12-31T18:00:00.000000000
2007-12-31T21:00:00.000000000


In [5]:
def hrdps_pca_num(k):
    print(f"Performing PCA with {k} components...")
    pressure = ds["pressure"]
    X_new = pressure.values.reshape(pressure.shape[0], -1)
    pca_hrdps = PCA(n_components=k)
    PCs_hrdps = pca_hrdps.fit_transform(X_new)
    print("Variance explained by each PC:")
    print(pca_hrdps.explained_variance_ratio_)
    print("Total variance explained:")
    print(pca_hrdps.explained_variance_ratio_.sum())
    print("-" * 50)

In [6]:
k_values = range(2, 25)
for k in k_values:
    hrdps_pca_num(k)

Performing PCA with 2 components...
Variance explained by each PC:
[0.8809278  0.06231838]
Total variance explained:
0.9432462
--------------------------------------------------
Performing PCA with 3 components...
Variance explained by each PC:
[0.8809278  0.06231841 0.03847013]
Total variance explained:
0.98171633
--------------------------------------------------
Performing PCA with 4 components...
Variance explained by each PC:
[0.88092774 0.06231836 0.0384701  0.00507361]
Total variance explained:
0.9867898
--------------------------------------------------
Performing PCA with 5 components...
Variance explained by each PC:
[0.88092774 0.06231835 0.03847007 0.00507361 0.00327741]
Total variance explained:
0.99006724
--------------------------------------------------
Performing PCA with 6 components...
Variance explained by each PC:
[0.88092774 0.06231843 0.03847007 0.00507361 0.0032774  0.00203845]
Total variance explained:
0.9921058
-------------------------------------------------

3 PCs explain 98% variance.

In [7]:
ds = xr.open_dataset("/home/dtaneja/analysis-dishika/notebooks/HRDPS_2007_pressure_with_latlon.nc")
lat_hr = ds["nav_lat"]
lon_hr = ds["nav_lon"]
corners = { "bottom_left": (float(lat_hr.isel(y=0, x=0)),float(lon_hr.isel(y=0, x=0))),
           "bottom_right": (float(lat_hr.isel(y=0, x=-1)),float(lon_hr.isel(y=0, x=-1))),
           "top_left": (float(lat_hr.isel(y=-1, x=0)),float(lon_hr.isel(y=-1, x=0))),
           "top_right": (float(lat_hr.isel(y=-1, x=-1)),float(lon_hr.isel(y=-1, x=-1)))}

for name, (lat, lon) in corners.items():
    print("name:", name)
    print("lat:", lat)
    print("lon:", lon)

hr_lats = []
hr_lons = []
for coord in corners.values():
    hr_lats.append(coord[0])
    hr_lons.append(coord[1])

lat_min = min(hr_lats)
lat_max = max(hr_lats)
lon_min = min(hr_lons)
lon_max = max(hr_lons)

print(lat_min, lat_max)
print(lon_min, lon_max)

name: bottom_left
lat: 45.61257553100586
lon: 232.6595001220703
name: bottom_right
lat: 46.535892486572266
lon: 240.79672241210938
name: top_left
lat: 51.41788864135742
lon: 230.60208129882812
name: top_right
lat: 52.4603157043457
lon: 239.75860595703125
45.61257553100586 52.4603157043457
230.60208129882812 240.79672241210938


In [9]:
files = sorted(glob.glob('/results/forcing/CanRCM5/*2007*.nc'))
files

['/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_huss.nc',
 '/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_pr.nc',
 '/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_psl.nc',
 '/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_rlds.nc',
 '/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_rsds.nc',
 '/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_tas.nc',
 '/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_uas.nc',
 '/results/forcing/CanRCM5/sc_nam22r55lkn3-bc07_200701_200712_3h_vas.nc']

In [11]:
files = sorted(glob.glob('/results/forcing/CanRCM5/*2007*.nc'))
for file in files:
    if "psl" in xr.open_dataset(file).data_vars: #Pressure
        ds = xr.open_dataset(file) 
        break

ds_2007 = ds.sel(time=slice("2007-01-03", "2007-12-31")) #Select data from 2007
print(ds_2007)
lat_lr = ds_2007["lat"]
lon_lr = ds_2007["lon"]

<xarray.Dataset> Size: 1GB
Dimensions:       (time: 2904, bnds: 2, rlat: 320, rlon: 360)
Coordinates:
  * time          (time) object 23kB 2007-01-03 00:00:00 ... 2007-12-31 21:00:00
  * rlat          (rlat) float64 3kB -35.11 -34.89 -34.67 ... 34.63 34.85 35.07
  * rlon          (rlon) float64 3kB -39.27 -39.05 -38.83 ... 39.27 39.49 39.71
    lon           (rlat, rlon) float64 922kB ...
    lat           (rlat, rlon) float64 922kB ...
Dimensions without coordinates: bnds
Data variables:
    time_bnds     (time, bnds) object 46kB ...
    rotated_pole  |S1 1B ...
    psl           (time, rlat, rlon) float32 1GB ...
Attributes: (12/19)
    CDI:                            Climate Data Interface version 2.0.3 (htt...
    Conventions:                    CF-1.4
    institution:                    CCCma (Canadian Centre for Climate Modell...
    title:                          CanRCM4 model output prepared for CORDEX ...
    institute_id:                   CCCma
    driving_experiment:      

In [12]:
lon_lr = ((lon_lr + 180) % 360) - 180
lon_min = ((lon_min + 180) % 360) - 180
lon_max = ((lon_max + 180) % 360) - 180

mask_lr = (
    (lat_lr >= lat_min) & 
    (lat_lr <= lat_max) &
    (lon_lr >= lon_min) & 
    (lon_lr <= lon_max)
)

i_idx, j_idx = np.where(mask_lr.values) # Finds array indices where mask_lr is True

i_min, i_max = i_idx.min(), i_idx.max()
j_min, j_max = j_idx.min(), j_idx.max()

ds_2007_cut = ds_2007.isel(rlat=slice(i_min, i_max + 1),rlon=slice(j_min, j_max + 1))
print(ds_2007_cut)

<xarray.Dataset> Size: 18MB
Dimensions:       (time: 2904, bnds: 2, rlat: 39, rlon: 39)
Coordinates:
  * time          (time) object 23kB 2007-01-03 00:00:00 ... 2007-12-31 21:00:00
  * rlat          (rlat) float64 312B 0.53 0.75 0.97 1.19 ... 8.45 8.67 8.89
  * rlon          (rlon) float64 312B -21.89 -21.67 -21.45 ... -13.75 -13.53
    lon           (rlat, rlon) float64 12kB 232.0 232.3 232.5 ... 239.3 239.6
    lat           (rlat, rlon) float64 12kB 43.66 43.74 43.82 ... 54.29 54.35
Dimensions without coordinates: bnds
Data variables:
    time_bnds     (time, bnds) object 46kB ...
    rotated_pole  |S1 1B ...
    psl           (time, rlat, rlon) float32 18MB ...
Attributes: (12/19)
    CDI:                            Climate Data Interface version 2.0.3 (htt...
    Conventions:                    CF-1.4
    institution:                    CCCma (Canadian Centre for Climate Modell...
    title:                          CanRCM4 model output prepared for CORDEX ...
    institute_id:  

In [13]:
output_file = "/home/dtaneja/analysis-dishika/notebooks/CanRCM5_2007_pressure_cut.nc"
ds_2007_cut.to_netcdf(output_file)

In [14]:
ds_cut = xr.open_dataset(output_file)
print(ds_cut)

<xarray.Dataset> Size: 18MB
Dimensions:       (time: 2904, bnds: 2, rlat: 39, rlon: 39)
Coordinates:
  * time          (time) object 23kB 2007-01-03 00:00:00 ... 2007-12-31 21:00:00
  * rlat          (rlat) float64 312B 0.53 0.75 0.97 1.19 ... 8.45 8.67 8.89
  * rlon          (rlon) float64 312B -21.89 -21.67 -21.45 ... -13.75 -13.53
    lon           (rlat, rlon) float64 12kB ...
    lat           (rlat, rlon) float64 12kB ...
Dimensions without coordinates: bnds
Data variables:
    time_bnds     (time, bnds) object 46kB ...
    rotated_pole  |S1 1B ...
    psl           (time, rlat, rlon) float32 18MB ...
Attributes: (12/19)
    CDI:                            Climate Data Interface version 2.0.3 (htt...
    Conventions:                    CF-1.4
    institution:                    CCCma (Canadian Centre for Climate Modell...
    title:                          CanRCM4 model output prepared for CORDEX ...
    institute_id:                   CCCma
    driving_experiment:             ,

In [15]:
pressure = ds_cut["psl"]
n_time = pressure.sizes["time"]
n_rlat = pressure.sizes["rlat"]
n_rlon = pressure.sizes["rlon"]
X_cut = pressure.values.reshape(n_time, n_rlat * n_rlon)

In [16]:
X_cut.shape

(2904, 1521)

In [17]:
pca_canrcm = PCA()
PCs_canrcm_all = pca_canrcm.fit_transform(X_cut)

cumulative_variance = np.cumsum(
    pca_canrcm.explained_variance_ratio_
)

for threshold in [0.90, 0.95, 0.98, 0.99]:
    n_pcs = np.argmax(cumulative_variance >= threshold) + 1
    print(f"{threshold * 100:.0f}% variance explained by {n_pcs} PCs")

90% variance explained by 2 PCs
95% variance explained by 3 PCs
98% variance explained by 4 PCs
99% variance explained by 5 PCs
